In [1]:
# ============================================
# 1. BASIS PENGETAHUAN
# ============================================

# Daftar penyakit (kode → nama)
penyakit = {
    "P01": "P01 - (isi nama penyakit)",
    "P02": "P02 - Dermatitis Atopik",
    "P03": "P03 - (isi nama penyakit)",
    "P04": "P04 - (isi nama penyakit)",
    "P05": "P05 - (isi nama penyakit)",
    "P06": "P06 - Herpes Zoster",
}

# (Opsional) Daftar gejala (kode → deskripsi), biar lebih mudah baca output
gejala = {
    "G01": "Ruam kemerahan",
    "G02": "Gatal",
    "G03": "Tekstur kering",
    "G04": "Bengkak",
    "G05": "Kulit bersisik",
    "G06": "Kulit melepuh",
    "G07": "Kulit menebal",
    "G08": "Kulit pecah-pecah",
    "G09": "Kulit terasa nyeri",
    "G10": "Penyakit menyebar",
    "G11": "Luka berair",
    "G13": "Adanya bercak",
    "G15": "Gejala lain (G15)",  # sesuaikan dengan PDF
    # Tambah gejala lain kalau ada
}

# Aturan: penyakit → daftar gejala yang relevan
# TODO: sesuaikan isi rules ini dengan tabel rule di PDF/materi
rules = {
    "P01": ["G01", "G02"],                      # contoh
    "P02": ["G01", "G02", "G05", "G07", "G15"], # isi sesuai tabel
    "P03": ["G03", "G04"],                      # contoh
    "P04": ["G06", "G07"],                      # contoh
    "P05": ["G08", "G09"],                      # contoh
    "P06": ["G02", "G05", "G06", "G09", "G11"], # isi sesuai tabel
}

# Bobot CF (MB, MD) untuk kombinasi (penyakit, gejala)
# TODO BESAR: isi nilai MB & MD sesuai tabel di PDF.
# Nilai di bawah ini hanya CONTOH STRUKTUR, bukan nilai asli.
cf_weights = {
    # ---------- Contoh P01 ----------
    ("P01", "G01"): (0.7, 0.1),
    ("P01", "G02"): (0.6, 0.2),

    # ---------- P02 (Dermatitis Atopik) ----------
    # Gantilah nilai angka-angka ini sesuai tabel CF di materi/PDF
    ("P02", "G01"): (0.8, 0.0),
    ("P02", "G02"): (0.8, 0.1),
    ("P02", "G05"): (0.7, 0.1),
    ("P02", "G07"): (0.7, 0.2),
    ("P02", "G15"): (0.6, 0.1),

    # ---------- Contoh P03 ----------
    ("P03", "G03"): (0.6, 0.1),
    ("P03", "G04"): (0.5, 0.2),

    # ---------- Contoh P04 ----------
    ("P04", "G06"): (0.7, 0.2),
    ("P04", "G07"): (0.6, 0.1),

    # ---------- Contoh P05 ----------
    ("P05", "G08"): (0.6, 0.1),
    ("P05", "G09"): (0.7, 0.2),

    # ---------- P06 (Herpes Zoster) ----------
    # Gantilah nilai angka-angka ini sesuai tabel CF di materi/PDF
    ("P06", "G02"): (0.7, 0.1),
    ("P06", "G05"): (0.8, 0.1),
    ("P06", "G06"): (0.9, 0.0),
    ("P06", "G09"): (0.8, 0.1),
    ("P06", "G11"): (0.9, 0.0),
}


# ============================================
# 2. FUNGSI PERHITUNGAN CERTAINTY FACTOR
# ============================================

def calculate_cf(mb, md):
    """
    Menghitung CF tunggal dari MB dan MD.
    CF = MB - MD
    """
    return mb - md


def combine_cf(cf1, cf2):
    """
    Mengkombinasikan dua nilai CF.
    Aturan:
    - Jika keduanya positif: CFcombine = cf1 + cf2 * (1 - cf1)
    - Jika keduanya negatif: CFcombine = cf1 + cf2 * (1 + cf1)
    - Jika tanda beda:       CFcombine = (cf1 + cf2) / (1 - min(|cf1|, |cf2|))
    """
    if cf1 >= 0 and cf2 >= 0:
        return cf1 + cf2 * (1 - cf1)
    elif cf1 < 0 and cf2 < 0:
        return cf1 + cf2 * (1 + cf1)
    else:
        return (cf1 + cf2) / (1 - min(abs(cf1), abs(cf2)))


# ============================================
# 3. MESIN INFERENSI
# ============================================

def run_inference(gejala_pasien):
    """
    Menjalankan mesin inferensi untuk menghitung CF setiap penyakit
    berdasarkan gejala yang dialami pasien.

    Parameter
    ---------
    gejala_pasien : list[str]
        Daftar kode gejala yang dialami pasien, misalnya ['G02', 'G05', 'G01'].

    Return
    ------
    list[tuple(str, float)]
        List tuple (nama_penyakit, cf_final), diurutkan dari CF tertinggi.
    """
    hasil_diagnosis = {}

    # Iterasi setiap penyakit dalam basis pengetahuan
    for kode_penyakit, nama_penyakit in penyakit.items():

        # Dapatkan daftar gejala yang relevan untuk penyakit ini
        gejala_relevan = rules.get(kode_penyakit, [])

        # Filter gejala pasien yang sesuai dengan aturan penyakit ini
        gejala_cocok = [g for g in gejala_pasien if g in gejala_relevan]

        # Jika tidak ada gejala yang cocok, lanjut ke penyakit berikutnya
        if not gejala_cocok:
            continue

        # Hitung CF untuk setiap gejala yang cocok
        cf_list = []
        for g in gejala_cocok:
            mb, md = cf_weights.get((kode_penyakit, g), (0, 0))

            # Pastikan ada bobotnya (kalau MB & MD = 0 semua, di-skip)
            if mb + md > 0:
                cf = calculate_cf(mb, md)
                cf_list.append(cf)

        # Jika tidak ada CF yang valid, lanjut
        if not cf_list:
            continue

        # Kombinasikan semua CF untuk penyakit ini
        cf_final = cf_list[0]
        for i in range(1, len(cf_list)):
            cf_final = combine_cf(cf_final, cf_list[i])

        # Simpan ke hasil diagnosis (pakai nama penyakit agar lebih jelas)
        hasil_diagnosis[nama_penyakit] = cf_final

    # Urutkan hasil dari CF tertinggi ke terendah
    hasil_urut = sorted(
        hasil_diagnosis.items(),
        key=lambda item: item[1],
        reverse=True
    )

    return hasil_urut


# ============================================
# 4. FUNGSI BANTU UNTUK MENAMPILKAN HASIL
# ============================================

def cetak_hasil(judul, hasil):
    print(judul)
    if not hasil:
        print("  (Tidak ada penyakit yang terdeteksi berdasarkan gejala ini.)")
        return

    for nama_penyakit, cf in hasil:
        print(f"  {nama_penyakit}: {cf * 100:.2f}%")
    print()  # baris kosong


# ============================================
# 5. UJI KASUS RESPONDEN
# ============================================

# ----- Responden 137 -----
# Dari teks tugas:
# Gatal (G02), Kulit bersisik (G05), Luka berair (G11),
# Ruam kemerahan (G01), Tekstur kering (G03),
# Kulit melepuh (G06), Kulit menebal (G07),
# Kulit pecah-pecah (G08), Kulit terasa nyeri (G09), Bengkak (G04)
input_137 = ["G02", "G05", "G11", "G01", "G03", "G06", "G07", "G08", "G09", "G04"]

# ----- Responden 144 -----
# Dari teks tugas:
# Gatal (G02), Ruam kemerahan (G01), Kulit menebal (G07),
# Kulit nyeri (G09), Penyakit menyebar (G10), Adanya bercak (G13)
input_144 = ["G02", "G01", "G07", "G09", "G10", "G13"]

# ----- Responden 179 -----
# Dari soal: ['G02', 'G05', 'G01', 'G15']
input_179 = ["G02", "G05", "G01", "G15"]


# Jalankan inferensi untuk tiap responden
hasil_137 = run_inference(input_137)
hasil_144 = run_inference(input_144)
hasil_179 = run_inference(input_179)

# Tampilkan hasil
cetak_hasil("--- Hasil Diagnosis Responden 137 ---", hasil_137)
cetak_hasil("--- Hasil Diagnosis Responden 144 ---", hasil_144)
cetak_hasil("--- Hasil Diagnosis Responden 179 ---", hasil_179)


--- Hasil Diagnosis Responden 137 ---
  P06 - Herpes Zoster: 99.96%
  P02 - Dermatitis Atopik: 98.80%
  P01 - (isi nama penyakit): 76.00%
  P04 - (isi nama penyakit): 75.00%
  P05 - (isi nama penyakit): 75.00%
  P03 - (isi nama penyakit): 65.00%

--- Hasil Diagnosis Responden 144 ---
  P02 - Dermatitis Atopik: 97.00%
  P06 - Herpes Zoster: 88.00%
  P01 - (isi nama penyakit): 76.00%
  P04 - (isi nama penyakit): 50.00%
  P05 - (isi nama penyakit): 50.00%

--- Hasil Diagnosis Responden 179 ---
  P02 - Dermatitis Atopik: 98.80%
  P06 - Herpes Zoster: 88.00%
  P01 - (isi nama penyakit): 76.00%

